# 1.0 imports

In [1]:
import pandas as pd
from sklearn.ensemble import IsolationForest
import joblib
import os

# 2.0 data loading

In [3]:
df = pd.read_csv(r'src\raw_data\Norway-NA-15_47_9-F-9 A time.csv')
df.head()

C:\Users\ABEL\AppData\Local\Temp\ipykernel_17548\830736183.py:1: DtypeWarning: Columns (119: Pass Name unitless, 146: Date of Chromatograph Sample unitless, 226: Time Time, 229: WBSCALIPERMNEMONIC unitless, 232: WBSBITDEPMNEMONIC unitless, 236: WBSHOLEDEPMNEMONIC unitless) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'src\raw_data\Norway-NA-15_47_9-F-9 A time.csv')


,Unnamed: 0.1,Unnamed: 0,Time s,Pump 2 Stroke Rate 1/min,MudPit Volume Average 5 m3,TOFB s,Block Velocity m/s,SPP/SPM2 unitless,MAG_DEC dega,Mud Density Out g/cm3,...,WBSCALIPERMNEMONIC unitless,WBSTRAJECTORYMNEMONIC unitless,WBSLITHOLOGY4 unitless,WBSBITDEPMNEMONIC unitless,WBSLITHOLOGY3 unitless,WBSLITHOLOGY2 unitless,WBSTUBULARMNEMONIC unitless,WBSHOLEDEPMNEMONIC unitless,WBSWBGEOMETRYMNEMONIC unitless,DateTime parsed
0,0,0.0,2009-06-27T16:50:29.000Z,NaN,61.529999,NaN,NaN,NaN,NaN,0.79,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009-06-27 16:50:29+00:00
1,1,1.0,2009-06-27T16:50:33.000Z,NaN,61.400002,NaN,NaN,NaN,NaN,0.79,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009-06-27 16:50:33+00:00
2,2,2.0,2009-06-27T16:50:38.000Z,NaN,61.290001,NaN,NaN,NaN,NaN,0.79,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009-06-27 16:50:38+00:00
3,3,3.0,2009-06-27T16:50:42.000Z,NaN,61.209999,NaN,NaN,NaN,NaN,0.79,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009-06-27 16:50:42+00:00
4,4,4.0,2009-06-27T16:50:47.000Z,NaN,61.220001,NaN,NaN,NaN,NaN,0.79,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009-06-27 16:50:47+00:00


# 3.0 data cleaning

In [4]:
chosen_df = df[["Time s", "Weight on Bit kkgf","Average Rotary Speed rpm","Average Surface Torque kN.m", "Average Standpipe Pressure kPa","MWD Stick-Slip PKtoPK RPM rpm","MWD Shock Peak m/s2"]]
chosen_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 419747 entries, 0 to 419746
Data columns (total 7 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Time s                          419745 non-null  str    
 1   Weight on Bit kkgf              86314 non-null   float64
 2   Average Rotary Speed rpm        86314 non-null   float64
 3   Average Surface Torque kN.m     85929 non-null   float64
 4   Average Standpipe Pressure kPa  85929 non-null   float64
 5   MWD Stick-Slip PKtoPK RPM rpm   15023 non-null   float64
 6   MWD Shock Peak m/s2             45914 non-null   float64
dtypes: float64(6), str(1)
memory usage: 22.4 MB


In [5]:
clean_df = chosen_df.copy()
clean_df = clean_df.dropna(subset=['Weight on Bit kkgf','Average Rotary Speed rpm'])
clean_df = clean_df[(clean_df['Weight on Bit kkgf'] > 0) & (clean_df['Average Rotary Speed rpm'] > 0)]
clean_df["MWD Stick-Slip PKtoPK RPM rpm"] = clean_df["MWD Stick-Slip PKtoPK RPM rpm"].ffill()
clean_df["MWD Shock Peak m/s2"] = clean_df["MWD Shock Peak m/s2"].ffill()
clean_df = clean_df.dropna()

In [6]:
clean_df.head()

,Time s,Weight on Bit kkgf,Average Rotary Speed rpm,Average Surface Torque kN.m,Average Standpipe Pressure kPa,MWD Stick-Slip PKtoPK RPM rpm,MWD Shock Peak m/s2
222592,2009-07-07T12:03:44.000Z,1.932303,126.0,6.413019,3957.108053,21.0,0.0
222594,2009-07-07T12:03:47.000Z,1.954983,126.0,6.860439,3930.563238,21.0,0.0
222595,2009-07-07T12:03:50.000Z,1.991271,125.0,7.023137,3948.765397,21.0,0.0
222596,2009-07-07T12:03:53.000Z,1.914160,125.0,7.023137,3948.765397,21.0,0.0
222598,2009-07-07T12:03:56.000Z,1.905088,125.0,7.050253,3948.420659,21.0,0.0


In [7]:
clean_df.info()

<class 'pandas.DataFrame'>
Index: 18761 entries, 222592 to 260096
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Time s                          18761 non-null  str    
 1   Weight on Bit kkgf              18761 non-null  float64
 2   Average Rotary Speed rpm        18761 non-null  float64
 3   Average Surface Torque kN.m     18761 non-null  float64
 4   Average Standpipe Pressure kPa  18761 non-null  float64
 5   MWD Stick-Slip PKtoPK RPM rpm   18761 non-null  float64
 6   MWD Shock Peak m/s2             18761 non-null  float64
dtypes: float64(6), str(1)
memory usage: 1.1 MB


# 4.0 Feature Engineering

In [8]:
# 01 Feature: Mechanical Specific Energy (MSE)
bit_size = 8.5 # assumes that the drill bit is 8.5 inches in diameter
clean_df['MSE'] = clean_df['Weight on Bit kkgf'] + (120 * clean_df['Average Rotary Speed rpm'] * clean_df['Average Surface Torque kN.m']) / (bit_size ** 2)

# 02 Feature: Instability (30 seconds Window)
clean_df['Torque_Variance'] = clean_df['Average Surface Torque kN.m'].rolling(window=30).std().fillna(0)
clean_df['Pressure_Variance'] = clean_df['Average Standpipe Pressure kPa'].rolling(window=30).std().fillna(0)

In [9]:
clean_df[['MSE','Torque_Variance','Pressure_Variance']].describe()

,MSE,Torque_Variance,Pressure_Variance
count,18761.000000,18761.000000,18761.000000
mean,2132.132972,0.473874,205.083401
std,911.801354,0.513361,464.212290
min,-384.740423,0.000000,0.000000
25%,1625.181033,0.299341,40.982856
50%,2093.785204,0.371000,64.477728
75%,2846.017245,0.469298,124.657072
max,4185.817933,6.500871,4185.295155


MSE can't be negative since it is illogical to produce negative energy, thus we will clean this up

In [10]:
clean_df = clean_df[(clean_df['MSE'] > 0) &
                    (clean_df['Weight on Bit kkgf'] > 0) & 
                    (clean_df['Average Surface Torque kN.m'] > 0)
                    ]
clean_df.describe()

,Weight on Bit kkgf,Average Rotary Speed rpm,Average Surface Torque kN.m,Average Standpipe Pressure kPa,MWD Stick-Slip PKtoPK RPM rpm,MWD Shock Peak m/s2,MSE,Torque_Variance,Pressure_Variance
count,18661.000000,18661.000000,18661.000000,18661.000000,18661.000000,18661.000000,18661.000000,18661.000000,18661.000000
mean,5.899658,154.232410,8.040352,7040.555731,38.397460,12.927925,2143.786863,0.473454,199.167996
std,4.372799,44.850114,2.144473,2971.463438,53.976188,35.252508,900.188607,0.506181,451.675712
min,0.004536,1.000000,0.013558,383.003768,0.000000,0.000000,1.357078,0.000000,0.000000
25%,2.122812,140.000000,7.904419,4797.854758,9.000000,0.000000,1642.830992,0.299990,40.935501
50%,4.486029,141.000000,8.663677,6288.225494,21.000000,0.000000,2096.585380,0.371151,64.346845
75%,10.614061,194.000000,9.192446,10226.441912,45.000000,24.000000,2850.481142,0.469179,122.556269
max,23.831743,235.000000,16.879933,11409.168578,381.000000,504.000000,4185.817933,6.484631,4185.295155


Now, as we can see on 'min' there are no negative values.

# 5.0 Machine Learning

In this stage, we will create a ML model to detect anomalies. Which means this model is able to predict or assume when the drill is about to break, thus operators could safely adjust the drill before it breaking.

## 5.1 Feature Selection

We have chose these features because:
* MSE = If the drill is spinning very fast, however it is not cutting anything it wastes energy. Thus, spiking the MSE.
* Torque & Pressure Variance = If the torque or pressure extremely swings back and forth (high variance) it indicates that the pipe is snapping.
* MWD Stick-Slip & Shock Peak = Shows how violently the drill vibrates

In [11]:
features = ['MSE', 'Torque_Variance', 'Pressure_Variance', 'MWD Stick-Slip PKtoPK RPM rpm', 'MWD Shock Peak m/s2']
X = clean_df[features]

Here, the model we will be using is IsolationForest. It is an unsupervised tree-based model which is highly capable on detecting anomalies by isolating them.

In [12]:
print("Training IsolationForest Model..")
iso_forest = IsolationForest(n_estimators=100,contamination=0.01,random_state=42)
iso_forest.fit(X)

Training IsolationForest Model..


,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.01
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",None
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


In [13]:
clean_df['Anomaly_Flag'] = iso_forest.predict(X)

In [14]:
print("\nAnomaly Counts (1 = Normal, -1 = Anomaly):")
print(clean_df['Anomaly_Flag'].value_counts())


Anomaly Counts (1 = Normal, -1 = Anomaly):
Anomaly_Flag
 1    18474
-1      187
Name: count, dtype: int64


## 5.2 Saving the Model

In [15]:
os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)

1. Save the model

In [16]:
model_path = "models/isolation_forest.pkl"
joblib.dump(iso_forest, model_path)
print(f"Model saved to {model_path}")

Model saved to models/isolation_forest.pkl


2. Save the cleaned data

In [17]:
export_cols = ['Time s','Average Surface Torque kN.m'] + features + ['Anomaly_Flag']
data_path = "data/rig_telemetry_drill.csv"
clean_df[export_cols].to_csv(data_path)
print(f"Data is saved to {data_path}")

Data is saved to data/rig_telemetry_drill.csv
